# LightGBM Model

Gradient-boosted decision-tree model using LightGBM.

Current benchmarks:

- Logistic Regression CV: **0.938096**
- CatBoost OOF CV: **0.941711**
- CatBoost Public LB: **0.941560**

Goal: determine whether LightGBM provides stronger or complementary predictions.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [3]:
categorical_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

for col in categorical_features:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [4]:
train[categorical_features].dtypes

Gender                    category
City_Type                 category
Current_Car_Type          category
Home_Charging_Possible    category
Subsidy_Available         category
Range_Anxiety_Level       category
dtype: object

In [5]:
X = train.drop(columns=["Will_Buy_EV", "id"])

y = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

X_test = test.drop(columns=["id"])

In [6]:
X[categorical_features].dtypes

Gender                    category
City_Type                 category
Current_Car_Type          category
Home_Charging_Possible    category
Subsidy_Available         category
Range_Anxiety_Level       category
dtype: object

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
X_train[categorical_features].dtypes

Gender                    category
City_Type                 category
Current_Car_Type          category
Home_Charging_Possible    category
Subsidy_Available         category
Range_Anxiety_Level       category
dtype: object

In [9]:
lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [10]:
lightgbm_model.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="auc",
    categorical_feature="auto",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=150,
            first_metric_only=True
        ),
        lgb.log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.940728	valid_0's binary_logloss: 0.228611
[400]	valid_0's auc: 0.94138	valid_0's binary_logloss: 0.227315
[600]	valid_0's auc: 0.941578	valid_0's binary_logloss: 0.226945
[800]	valid_0's auc: 0.941649	valid_0's binary_logloss: 0.22682
[1000]	valid_0's auc: 0.941662	valid_0's binary_logloss: 0.226784
Early stopping, best iteration is:
[979]	valid_0's auc: 0.941669	valid_0's binary_logloss: 0.226775
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [11]:
lightgbm_valid_predictions = lightgbm_model.predict_proba(
    X_valid,
    num_iteration=lightgbm_model.best_iteration_
)[:, 1]

lightgbm_auc = roc_auc_score(
    y_valid,
    lightgbm_valid_predictions
)

lightgbm_auc

0.9416689105035138

In [12]:
lightgbm_model.best_iteration_

979

In [13]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": lightgbm_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance

,Feature,Importance
1,Annual_Income_USD,8542
2,Daily_Commute_km,5612
0,Age,4028
5,Charging_Stations_Near_Work,2590
4,Charging_Stations_Near_Home,2364
6,Environmental_Concern_Level,1932
12,Range_Anxiety_Level,926
3,Number_of_Cars_Owned,854
9,Current_Car_Type,601
11,Subsidy_Available,542


In [14]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "CatBoost",
        "LightGBM"
    ],
    "Single_Split_AUC": [
        0.9379578769,
        0.9414339188,
        lightgbm_auc
    ]
})

comparison.sort_values(
    "Single_Split_AUC",
    ascending=False
)

,Model,Single_Split_AUC
2,LightGBM,0.941669
1,CatBoost,0.941434
0,Logistic Regression,0.937958


In [15]:
from sklearn.model_selection import StratifiedKFold

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions_lgb = np.zeros(len(X))
test_predictions_lgb = np.zeros(len(X_test))

fold_scores_lgb = []
best_iterations_lgb = []

In [17]:
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    print(f"\n{'=' * 50}")
    print(f"FOLD {fold}")
    print(f"{'=' * 50}")

    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    model = LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42 + fold,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        eval_X=X_valid_fold,
        eval_y=y_valid_fold,
        eval_metric="auc",
        categorical_feature="auto",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                first_metric_only=True,
                verbose=False
            )
        ]
    )

    valid_pred = model.predict_proba(
        X_valid_fold,
        num_iteration=model.best_iteration_
    )[:, 1]

    test_pred = model.predict_proba(
        X_test,
        num_iteration=model.best_iteration_
    )[:, 1]

    oof_predictions_lgb[valid_idx] = valid_pred
    test_predictions_lgb += test_pred / 5

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_pred
    )

    fold_scores_lgb.append(fold_auc)
    best_iterations_lgb.append(
        model.best_iteration_
    )

    print(f"Fold {fold} AUC: {fold_auc:.6f}")
    print(f"Best iteration: {model.best_iteration_}")


FOLD 1
Fold 1 AUC: 0.940614
Best iteration: 1292

FOLD 2
Fold 2 AUC: 0.941525
Best iteration: 1109

FOLD 3
Fold 3 AUC: 0.942845
Best iteration: 1160

FOLD 4
Fold 4 AUC: 0.942252
Best iteration: 767

FOLD 5
Fold 5 AUC: 0.941670
Best iteration: 977


In [18]:
oof_auc_lgb = roc_auc_score(
    y,
    oof_predictions_lgb
)

print("\nFold scores:")

for fold, score in enumerate(fold_scores_lgb, start=1):
    print(f"Fold {fold}: {score:.6f}")

print()
print(f"Mean fold AUC: {np.mean(fold_scores_lgb):.6f}")
print(f"OOF AUC:       {oof_auc_lgb:.6f}")
print(f"Std AUC:       {np.std(fold_scores_lgb):.6f}")

print()
print("Best iterations:")
print(best_iterations_lgb)


Fold scores:
Fold 1: 0.940614
Fold 2: 0.941525
Fold 3: 0.942845
Fold 4: 0.942252
Fold 5: 0.941670

Mean fold AUC: 0.941781
OOF AUC:       0.941770
Std AUC:       0.000748

Best iterations:
[1292, 1109, 1160, 767, 977]


In [19]:
oof_lgb_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "lightgbm_oof": oof_predictions_lgb
})

oof_lgb_df.to_csv(
    "../outputs/predictions/lightgbm_oof.csv",
    index=False
)

In [20]:
test_lgb_df = pd.DataFrame({
    "id": test["id"],
    "lightgbm_prediction": test_predictions_lgb
})

test_lgb_df.to_csv(
    "../outputs/predictions/lightgbm_test.csv",
    index=False
)

In [21]:
submission_lgb = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions_lgb
})

submission_lgb.head()

,id,Will_Buy_EV
0,668665,0.010252
1,668666,0.017503
2,668667,0.005768
3,668668,0.003677
4,668669,0.018970


In [22]:
submission_lgb.shape

(286571, 2)

In [23]:
submission_lgb.isna().sum()

id             0
Will_Buy_EV    0
dtype: int64

In [24]:
submission_lgb["id"].equals(test["id"])

True

In [25]:
submission_lgb.to_csv(
    "../outputs/submissions/submission_002_lightgbm_cv.csv",
    index=False
)